# Practice: softmax, cross-entropy and a 10-digit classifier
*AI from scratch, chapter 5* · lesson: https://ihor.sh/ai/05-probability-and-loss.html

A safe softmax, cross-entropy and its famous gradient (checked against PyTorch), softmax regression that reads
handwritten digits, the same model in PyTorch, and sampling with temperature.
Fill in each `...` under `# YOUR CODE HERE`, then run the check under it (✓ = right, ✗ = not yet).

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ["E.sum(axis=1, keepdims=True) is each row's total, kept as a column so the division lines up row by row.", 'return E / E.sum(axis=1, keepdims=True)'], 2: ['Compute predicted probabilities P = softmax(S).', 'Index true class probabilities: P[np.arange(len(y)), y].', 'Return negative mean of log probabilities: -np.mean(np.log(...)).'], 3: ['Compute predicted probabilities P = softmax(S).', 'Form one-hot matrix Y using np.eye(S.shape[1])[y].', 'Return (P - Y) / len(y).'], 4: ['Run 300 steps of gradient descent: Compute logits S = Xtr @ W + b. Compute gradient G = ce_grad(S, ytr). Update W -= 0.5 * Xtr.T @ G. Update b -= 0.5 * G.sum(axis=0).', 'Compute test accuracy: ((Xte @ W + b).argmax(axis=1) == yte).mean().'], 5: ['Run 300 training steps: opt.zero_grad(); loss_fn(model(Xt), yt).backward(); opt.step().', 'Inside torch.no_grad(), compute test accuracy torch_acc.'], 6: ['Sample 10,000 times at T=0.5 with rng.choice(6, 10_000, p=softmax(logits, 0.5)[0]).', 'Compute share_top_cold as fraction where index 0 ("mat") was sampled.', 'Sample 10,000 times at T=2.0 with rng.choice(6, 10_000, p=softmax(logits, 2.0)[0]).', 'Compute share_top_hot as fraction where index 0 was sampled.']}
_ANSWERS = {1: 'cmV0dXJuIEUgLyBFLnN1bShheGlzPTEsIGtlZXBkaW1zPVRydWUp', 2: 'UCA9IHNvZnRtYXgoUykKcmV0dXJuIC1ucC5tZWFuKG5wLmxvZyhQW25wLmFyYW5nZShsZW4oeSkpLCB5XSkp', 3: 'cmV0dXJuIChzb2Z0bWF4KFMpIC0gbnAuZXllKFMuc2hhcGVbMV0pW3ldKSAvIGxlbih5KQ==', 4: 'Zm9yIHN0ZXAgaW4gcmFuZ2UoMzAwKToKICAgIEcgPSBjZV9ncmFkKFh0ciBAIFcgKyBiLCB5dHIpCiAgICBXIC09IDAuNSAqIFh0ci5UIEAgRwogICAgYiAtPSAwLjUgKiBHLnN1bShheGlzPTApCnRlc3RfYWNjID0gKChYdGUgQCBXICsgYikuYXJnbWF4KGF4aXM9MSkgPT0geXRlKS5tZWFuKCk=', 5: 'Zm9yIHN0ZXAgaW4gcmFuZ2UoMzAwKToKICAgIG9wdC56ZXJvX2dyYWQoKTsgbG9zc19mbihtb2RlbChYdCksIHl0KS5iYWNrd2FyZCgpOyBvcHQuc3RlcCgp', 6: 'c2hhcmVfdG9wX2NvbGQgPSAocm5nLmNob2ljZSg2LCAxMF8wMDAsIHA9c29mdG1heChsb2dpdHMsIDAuNSlbMF0pID09IDApLm1lYW4oKQpzaGFyZV90b3BfaG90ID0gKHJuZy5jaG9pY2UoNiwgMTBfMDAwLCBwPXNvZnRtYXgobG9naXRzLCAyLjApWzBdKSA9PSAwKS5tZWFuKCk='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

## 1. A softmax that never overflows
`softmax(S, T=1)` takes a 2D array of scores (one row per example). Its first two lines are written: divide by the
temperature T, then exponentiate after subtracting each row's max (the lesson's trick: the biggest becomes e⁰ = 1,
so nothing overflows). Finish it: divide `E` by each row's sum, with `axis=1, keepdims=True` as in the line above.

In [ ]:
def softmax(S, T=1.0):
    """Numerically stable softmax activation with temperature scaling.
    
    Formula: exp((S - max(S)) / T) / sum(exp((S - max(S)) / T))
    
    Args:
        S: 2D array-like of logits/scores with shape (n_samples, n_classes).
        T: Float temperature parameter (default 1.0).
    Returns:
        NumPy array of probabilities of shape (n_samples, n_classes), summing to 1 along axis 1.
    """
    S = np.asarray(S, dtype=float) / T
    E = np.exp(S - S.max(axis=1, keepdims=True))
    # YOUR CODE HERE (stuck? hint(1), then answer(1))
    ...

In [ ]:
check(1, lambda: np.allclose(softmax([[2, 1, -0.5]]), [[0.690, 0.254, 0.057]], atol=1e-3), 
      "softmax([[2, 1, -0.5]]) should produce probabilities ~[0.690, 0.254, 0.057].")
check(1, lambda: np.allclose(softmax([[1000, 999]]), [[0.731, 0.269]], atol=1e-3), 
      "Softmax must be numerically stable and avoid overflow even with large logits like 1000.")
check(1, lambda: softmax([[2, 1, -0.5]], T=0.1)[0, 0] > 0.99 and softmax([[2, 1, -0.5]], T=10)[0, 0] < 0.4, 
      "Low temperature (T=0.1) should concentrate probability on the top class, high T (T=10) should flatten it.")

## 2. Cross-entropy and its gradient
Write `cross_entropy(S, y)`: the mean of $-\ln p_{i, y_i}$ over the rows, and `ce_grad(S, y)`: the gradient with
respect to the scores, $(P - Y)/n$ with Y the one-hot labels. PyTorch's `F.cross_entropy` and autograd check you.

In [ ]:
def cross_entropy(S, y):
    """Compute multiclass cross-entropy loss from raw logits S and integer labels y.
    
    Formula: -mean(ln(P[i, y_i]))
    
    Args:
        S: 2D array of logits with shape (n_samples, n_classes).
        y: 1D array of integer class labels (0 <= y_i < n_classes).
    Returns:
        float: Mean cross-entropy loss.
    """
    # YOUR CODE HERE (stuck? hint(2), then answer(2))
    ...

def ce_grad(S, y):
    """Compute gradient of cross-entropy loss with respect to input logits S.
    
    Formula: (P - Y) / n, where P = softmax(S) and Y is one-hot encoded targets.
    
    Args:
        S: 2D array of logits with shape (n_samples, n_classes).
        y: 1D array of integer class labels (shape n_samples,).
    Returns:
        NumPy array of shape (n_samples, n_classes) representing dL/dS.
    """
    # YOUR CODE HERE (stuck? hint(3), then answer(3))
    ...

S = np.random.default_rng(0).normal(size=(5, 4)); y = np.array([0, 3, 1, 1, 2])
St = torch.tensor(S, requires_grad=True)
loss_t = F.cross_entropy(St, torch.tensor(y)); loss_t.backward()

In [ ]:
check(3, lambda: np.isclose(cross_entropy(S, y), loss_t.item()), 
      "Your cross_entropy(S, y) must match PyTorch's F.cross_entropy.")
check(3, lambda: np.allclose(ce_grad(S, y), St.grad.numpy()), 
      "ce_grad(S, y) must match PyTorch autograd St.grad ((P - Y)/n).")

## 3. Read handwritten digits
scikit-learn's 1,797 digit images (8×8 = 64 pixels, values 0–16). Train **softmax regression** (one linear layer:
10 scores) by gradient descent: `W` is (64, 10), `b` is (10,); 300 steps, learning rate 0.5, using your
`ce_grad` (the gradient for W is $X^\top G$, for b the column sums of G). Set `test_acc`.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
dg = load_digits()
Xtr, Xte, ytr, yte = train_test_split(dg.data / 16, dg.target, test_size=0.25, random_state=0)
W, b = np.zeros((64, 10)), np.zeros(10)
test_acc = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print("test accuracy:", test_acc)

In [ ]:
check(4, lambda: test_acc > 0.9, 
      "Softmax regression on 8x8 digits should exceed 90% test accuracy.")

## 4. The same model in PyTorch
`nn.Linear(64, 10)` + `nn.CrossEntropyLoss()` (which applies the softmax for you: pass raw scores, "logits").
Train with SGD, lr 0.5, 300 full-batch steps. Set `torch_acc`.

In [ ]:
from torch import nn
torch.manual_seed(0)
Xt, yt = torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr)
model, loss_fn = nn.Linear(64, 10), nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.5)
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
with torch.no_grad():
    torch_acc = (model(torch.tensor(Xte, dtype=torch.float32)).argmax(1).numpy() == yte).mean()
print("PyTorch:", torch_acc)

In [ ]:
check(5, lambda: torch_acc > 0.9, 
      "PyTorch softmax regression should achieve over 90% test accuracy.")

## 5. Sampling with temperature
A language model's next-word scores. Draw 10,000 samples at T = 0.5 and at T = 2 (`rng.choice(len(words), p=...)`)
and set `share_top_cold` and `share_top_hot`: how often the most likely word ("mat") comes out at each.

In [ ]:
words = ["mat", "floor", "chair", "roof", "keyboard", "moon"]
logits = np.array([[3.1, 2.2, 1.6, 0.9, 1.2, -1.5]])
rng = np.random.default_rng(1)
share_top_cold = share_top_hot = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...
print(share_top_cold, share_top_hot)

In [ ]:
check(6, lambda: share_top_cold > 0.75 and share_top_hot < 0.4, 
      "Cold temperature (T=0.5) should sample the top token > 75% of the time; hot temperature (T=2.0) drops it below 40%.")

## Next
Back to the path: https://ihor.sh/ai/ · next: calculus 4, integrals.